In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
print(train.head())
train.shape

   id                                             prompt  \
0   1  Pick the best possible answer: What is Martin ...   
1   2        What is accelerator-based light-ion fusion?   
2   3  Determine the correct option: What is the term...   
3   4  Select the most accurate option: What is Marti...   
4   5  Identify the correct statement: What is the co...   

                                                   A  \
0  Martin Heidegger believes that humans exist wi...   
1  Accelerator-based light-ion fusion is a techni...   
2                                       Blueshifting   
3  Martin Heidegger believes that humans exist wi...   
4  Simultaneity is relative, meaning that two eve...   

                                                   B  \
0  Martin Heidegger believes that humans do not e...   
1  Accelerator-based light-ion fusion is a techni...   
2                                        Redshifting   
3  Martin Heidegger believes that humans do not e...   
4  Simultaneity is rel

(2000, 8)

QUESTION_1:Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?  


In [3]:
v = train["answer"].value_counts()
print(v)
highest_v = v.max()
lowest_v = v.min()
result = highest_v + lowest_v
print("max", highest_v)
print("min", lowest_v)
print(result)

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
max 490
min 324
814


QUESTION_2:After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv? 

In [4]:
import string

all_words = set()

for text in train["prompt"]:
    text = str(text).lower()
    text = text.translate(
        str.maketrans('','', string.punctuation)
    )
    words = text.split()
    all_words.update(words)

print("vocab_size", len(all_words))

vocab_size 859


QUESTION_3:Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?  

In [5]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
prompt = train.iloc[0]["prompt"]
prompt = prompt.lower()
prompt = prompt.translate(
    str.maketrans('','', string.punctuation)
)

words = prompt.split()
filtered_words = [
    word for word in words
    if  word not in ENGLISH_STOP_WORDS
]

print(filtered_words)
print("count", len(filtered_words))

['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']
count 13


QUESTION_4:Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?  

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

combined_text = (
    train['prompt'].fillna('') + ' ' +
    train['A'].fillna('') + ' ' +
    train['B'].fillna('') + ' ' +
    train['C'].fillna('') + ' ' +
    train['D'].fillna('') + ' ' +
    train['E'].fillna('')
)

vectorizer = TfidfVectorizer(
    stop_words='english'
)

X = vectorizer.fit_transform(combined_text)

print("Vocabulary Size =", len(vectorizer.vocabulary_))

Vocabulary Size = 2762


QUESTION_5:Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).  

In [7]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Row ID 1 (first row)
prompt = train.iloc[0]['prompt']
option_a = train.iloc[0]['A']

# Fit TF-IDF on all combined text (same as Q4)
combined_text = (
    train['prompt'].fillna('') + ' ' +
    train['A'].fillna('') + ' ' +
    train['B'].fillna('') + ' ' +
    train['C'].fillna('') + ' ' +
    train['D'].fillna('') + ' ' +
    train['E'].fillna('')
)

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_text)

prompt_vec = vectorizer.transform([prompt])
a_vec = vectorizer.transform([option_a])

sim = cosine_similarity(prompt_vec, a_vec)[0][0]

print("Cosine Similarity =", round(sim, 4))

Cosine Similarity = 0.272


QUESTION_6:Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.   

In [8]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

combined_text = (
    train['prompt'].fillna('') + ' ' +
    train['A'].fillna('') + ' ' +
    train['B'].fillna('') + ' ' +
    train['C'].fillna('') + ' ' +
    train['D'].fillna('') + ' ' +
    train['E'].fillna('')
)

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_text)

correct = 0

for _, row in train.iterrows():

    prompt_vec = vectorizer.transform([row['prompt']])

    scores = []

    for opt in ['A','B','C','D','E']:

        option_vec = vectorizer.transform([row[opt]])

        score = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

        scores.append(score)

    pred = ['A','B','C','D','E'][scores.index(max(scores))]

    if pred == row['answer']:
        correct += 1

accuracy = 100 * correct / len(train)

print("Percentage =", round(accuracy, 2))

Percentage = 13.55


QUESTION_7:If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?  



In [9]:
actual = 'C'
preds = ['C','A','B']

if actual in preds:
    rank = preds.index(actual) + 1
    map3 = 1/rank
else:
    map3 = 0

print(map3)

1.0


QUESTION_8:If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E?  


In [10]:
actual = 'B'
preds = ['D','B','E']

if actual in preds:
    rank = preds.index(actual) + 1
    map3 = 1/rank
else:
    map3 = 0

print(map3)

0.5


QUESTION_9:The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [11]:
freq = train['answer'].value_counts()

print(freq)

top3 = list(freq.index[:3])

print("Static Prediction =", top3)

scores = []

for actual in train['answer']:

    if actual in top3:

        rank = top3.index(actual) + 1
        scores.append(1/rank)

    else:

        scores.append(0)

map3 = sum(scores)/len(scores)

print("MAP@3 =", round(map3,5))

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Static Prediction = ['B', 'C', 'A']
MAP@3 = 0.42125


QUESTION_10:The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?  

In [12]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity



# Fit TF-IDF on all text
combined_text = (
    train['prompt'].fillna('') + ' ' +
    train['A'].fillna('') + ' ' +
    train['B'].fillna('') + ' ' +
    train['C'].fillna('') + ' ' +
    train['D'].fillna('') + ' ' +
    train['E'].fillna('')
)

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_text)

options = ['A', 'B', 'C', 'D', 'E']

map3_scores = []

for _, row in train.iterrows():

    prompt_vec = vectorizer.transform([row['prompt']])

    similarities = []

    for opt in options:
        opt_vec = vectorizer.transform([row[opt]])
        sim = cosine_similarity(prompt_vec, opt_vec)[0][0]
        similarities.append((opt, sim))

    # Sort by similarity descending
    similarities.sort(key=lambda x: x[1], reverse=True)

    top3 = [x[0] for x in similarities[:3]]

    actual = row['answer']

    if actual in top3:
        rank = top3.index(actual) + 1
        score = 1 / rank
    else:
        score = 0

    map3_scores.append(score)

final_map3 = sum(map3_scores) / len(map3_scores)

print("TF-IDF Pipeline MAP@3 =", round(final_map3, 5))

TF-IDF Pipeline MAP@3 = 0.29617
